In [1]:
import os
import shutil
import time
from pathlib import Path

import cv2
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix, precision_recall_curve

from anomalib.data import Folder
from anomalib.engine import Engine
from anomalib.models import Patchcore
from lightning.pytorch.callbacks import TQDMProgressBar
from anomalib.data.utils.split import ValSplitMode, TestSplitMode

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

PyTorch version : 2.11.0+cpu
CUDA available  : False


In [2]:
# ── Dataset ────────────────────────────────────────────────────────────

DATASET_ROOT = r"C:\Users\Inga\miniconda3\envs\capstone\capstone_xray\data\xray\anomalib_data\Machine_S"
DATASET_NAME = ""

# ── Model ──────────────────────────────────────────────────────────────
BACKBONE       = "resnet18"
LAYERS         = ["layer2", "layer3"]
NUM_NEIGHBORS  = 9
CORESET_RATIO = 0.01

# ── DataLoader ─────────────────────────────────────────────────────────
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE  = 1
NUM_WORKERS      = 0

# ── Output ─────────────────────────────────────────────────────────────
#SAVE_DIR = Path("results/final_pipeline")
#SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Benchmarking ───────────────────────────────────────────────────────
# All latency measurements are performed exclusively on CPU.
NUM_RUNS           = 10
BENCH_WARMUP_RUNS  = 20    # warm-up passes (discarded)
BENCH_MEASURE_RUNS = 200   # timed passes
BENCH_IMAGE_SIZE   = (1, 3, 256, 256)  # (N, C, H, W) — must match training resolution

print("Configuration loaded.")

Configuration loaded.


In [3]:

accelerator_type = "gpu" if torch.cuda.is_available() else "cpu"
engine = Engine(
    accelerator=accelerator_type,
    devices=1,
    callbacks=[TQDMProgressBar(refresh_rate=0)],
)

print(f"Model      : PatchCore  |  Backbone: {BACKBONE}")
print(f"Layers     : {LAYERS}")
print(f"Neighbors  : {NUM_NEIGHBORS}")
print(f"Accelerator: {accelerator_type}")

Model      : PatchCore  |  Backbone: resnet18
Layers     : ['layer2', 'layer3']
Neighbors  : 9
Accelerator: cpu


In [4]:
def run_single_experiment(run_num, base_save_dir, dataset_root, backbone, layers, num_neighbors, 
                          coreset_sampling_ratio, train_batch_size, eval_batch_size, num_workers, 
                          bench_warmup_runs, bench_measure_runs, bench_image_size):
    
    # Setup unique save directory
    run_save_dir = base_save_dir / f"run_{run_num}"
    if run_save_dir.exists():
        shutil.rmtree(run_save_dir)
    run_save_dir.mkdir(parents=True, exist_ok=True)
    
    # DataModule
    datamodule = Folder(
        root=DATASET_ROOT,
        name="",
        normal_dir="train/good",
        abnormal_dir="test/bad",
        normal_test_dir="test/good",
        train_batch_size=32,
        eval_batch_size=1,
        num_workers=0,
        val_split_mode=ValSplitMode.SAME_AS_TEST,
    )
    
    datamodule.setup()
    
    # Model & Engine
    model = Patchcore(
        backbone=BACKBONE,
        num_neighbors=NUM_NEIGHBORS,
        pre_trained=True,
        coreset_sampling_ratio=CORESET_RATIO
    )

    
    engine = Engine(accelerator="auto", devices=1, default_root_dir=str(run_save_dir),
                    callbacks=[TQDMProgressBar(refresh_rate=0)])
    
    # Train & Test
    engine.fit(model=model, datamodule=datamodule)
    test_results = engine.test(model=model, datamodule=datamodule)
    metrics = test_results[0]
    
    # Threshold Selection
    preds = engine.predict(model=model, datamodule=datamodule)
    y_true = np.concatenate([getattr(b, "label", b.gt_label).cpu().numpy() for b in preds])
    y_scores = np.concatenate([getattr(b, "pred_scores", b.pred_score).cpu().numpy() for b in preds])
    
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1 = 2 * recall * precision / (recall + precision + 1e-10)
    best_thresh = thresholds[np.argmax(f1)]
    
    # Confusion Matrix
    y_pred = (y_scores >= best_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Latency Benchmarking
    model_cpu = model.eval().to("cpu")
    dummy = torch.randn(*BENCH_IMAGE_SIZE)
    with torch.no_grad():
        for _ in range(BENCH_WARMUP_RUNS): model_cpu(dummy)
        latencies = []
        for _ in range(BENCH_MEASURE_RUNS):
            start = time.perf_counter()
            model_cpu(dummy)
            latencies.append((time.perf_counter() - start) * 1000)
            
    return {
        "Run": run_num, "Backbone": backbone, "Coreset": coreset_sampling_ratio,
        "AUROC": metrics.get('image_AUROC', 0), "F1": np.max(f1), "Threshold": best_thresh,
        "FP": fp, "FN": fn, "FP_Rate": (fp / (fp + tn + 1e-10)) * 100,
        "FN_Rate": (fn / (fn + tp + 1e-10)) * 100, "Latency": np.mean(latencies)
    }

In [5]:
all_results = []
base_save_dir = Path(r"C:\Users\Inga\miniconda3\envs\capstone\capstone_xray\notebooks\results\ResNet18_experiments")

for i in range(1, NUM_RUNS + 1):
    print(f"Executing Run {i}/{NUM_RUNS}...")
    metrics = run_single_experiment(
        i, base_save_dir, DATASET_ROOT, BACKBONE, LAYERS, NUM_NEIGHBORS, 
        CORESET_RATIO, TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE, NUM_WORKERS,
        BENCH_WARMUP_RUNS, BENCH_MEASURE_RUNS, BENCH_IMAGE_SIZE
    )
    all_results.append(metrics)

# Generate Table & CSV
df = pd.DataFrame(all_results)
df.to_csv(base_save_dir / "all_experiments_summary.csv", index=False)
print(df.to_string(index=False))

Executing Run 1/10...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\core\optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │  2.8 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\loops\fit_loop.py:534: Found 69 module(s) in eval mode at the start of tra

...

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\loops\fit_loop.py:534: Found 69 module(s) in eval mode at the start of tra

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.9705244302749634     │
│       image_F1Score       │    0.9481765627861023     │
└───────────────────────────┴───────────────────────────┘

ckpt_path is not provided. Model weights will not be loaded.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


NameError: name 'pd' is not defined

In [6]:
import pandas as pd
all_results.append(metrics)

# Generate Table & CSV
df = pd.DataFrame(all_results)
df.to_csv(base_save_dir / "ResNet18_0.01_experiments_summary.csv", index=False)
print(df.to_string(index=False))

 Run Backbone  Coreset    AUROC       F1  Threshold  FP  FN   FP_Rate  FN_Rate    Latency
   1 resnet18     0.01 0.982066 0.972441        0.5   6   8  4.878049 3.137255 191.556418
   2 resnet18     0.01 0.986450 0.961538        0.5  15   5 12.195122 1.960784 143.613771
   3 resnet18     0.01 0.954535 0.951417        0.5   4  20  3.252033 7.843137 172.730408
   4 resnet18     0.01 0.960242 0.953535        0.5   4  19  3.252033 7.450980 153.752861
   5 resnet18     0.01 0.974781 0.960784        0.5  10  10  8.130081 3.921569 164.053599
   6 resnet18     0.01 0.974494 0.962818        0.5  10   9  8.130081 3.529412 168.408155
   7 resnet18     0.01 0.986737 0.957198        0.5  13   9 10.569106 3.529412 152.266436
   8 resnet18     0.01 0.977714 0.966469        0.5   7  10  5.691057 3.921569 149.414206
   9 resnet18     0.01 0.983293 0.953488        0.5  15   9 12.195122 3.529412 153.918558
  10 resnet18     0.01 0.970524 0.950192        0.5  19   7 15.447154 2.745098 153.722253
  10 resne